# Personalized Restaurant Recommendation System

A machine learning pipeline to predict which restaurants a customer is likely to order from,
based on **customer demographics**, **vendor attributes**, and **location proximity**.

### Pipeline Overview
1. Load & explore data
2. Build a proper **target column** by aggregating orders at the (customer, vendor) pair level
3. Generate balanced negative samples (customers who did NOT order from a vendor)
4. Merge only **prediction-time-available** features (no order-specific leakage)
5. Train an XGBClassifier with cross-validation
6. Predict on the test set & create submission

## 1. Load Datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from warnings import filterwarnings
filterwarnings("ignore")

# --- Load all datasets ---
train_customers = pd.read_csv("Datasets/Train/train_customers.csv")
train_locations = pd.read_csv("Datasets/Train/train_locations.csv")
orders = pd.read_csv("Datasets/Train/orders.csv", low_memory=False)
vendors = pd.read_csv("Datasets/Train/vendors.csv")

test_customers = pd.read_csv("Datasets/Test/test_customers.csv")
test_locations = pd.read_csv("Datasets/Test/test_locations.csv")

# Deduplicate customers (some have duplicate rows in the dataset)
train_customers = train_customers.drop_duplicates(subset='customer_id', keep='first')
test_customers = test_customers.drop_duplicates(subset='customer_id', keep='first')

print("=== Dataset Shapes ===")
print(f"Train Customers : {train_customers.shape}")
print(f"Train Locations : {train_locations.shape}")
print(f"Orders          : {orders.shape}")
print(f"Vendors         : {vendors.shape}")
print(f"Test Customers  : {test_customers.shape}")
print(f"Test Locations  : {test_locations.shape}")

## 2. Exploratory Data Analysis

In [ ]:
# --- Orders overview ---
print("=== Orders Columns ===")
print(orders.columns.tolist())

print("\n=== Orders per Customer ===")
print(orders.groupby('customer_id').size().describe())

print("\n=== Orders per Vendor ===")
print(orders.groupby('vendor_id').size().describe())

# --- Customer-Vendor pair analysis ---
cv_counts = orders.groupby(['customer_id', 'vendor_id']).size()
print("\n=== Customer-Vendor Pair Frequency ===")
print(cv_counts.describe())
print(f"\nUnique pairs:         {len(cv_counts):,}")
print(f"Pairs with 1 order:   {(cv_counts == 1).sum():,}")
print(f"Pairs with 2+ orders: {(cv_counts >= 2).sum():,}")
print(f"Pairs with 5+ orders: {(cv_counts >= 5).sum():,}")

# --- Train/Test overlap check ---
train_cids = set(train_customers['customer_id'])
test_cids = set(test_customers['customer_id'])
overlap = train_cids & test_cids
print(f"\n=== Train/Test Customer Overlap ===")
print(f"Train customers: {len(train_cids):,}")
print(f"Test customers:  {len(test_cids):,}")
print(f"Overlap:         {len(overlap)} (cold-start scenario!)")

## 3. Build Proper Target Column

**Key insight**: Instead of setting `target = 1` for every individual order row (which gives 135K rows
all labeled 1), we **aggregate to the (customer, vendor) pair level**.

- Each unique (customer, vendor) pair that had at least one order -> `target = 1` (positive)
- Customer-vendor pairs that never interacted -> `target = 0` (negative, generated next)

This collapses ~135K order rows into ~71K unique interaction pairs, removing duplicate noise.

In [ ]:
# --- Aggregate orders at the (customer, vendor) level ---
cv_positives = orders.groupby(['customer_id', 'vendor_id']).agg(
    order_count=('order_id', 'count'),
).reset_index()

# Binary target: 1 = customer ordered from this vendor
cv_positives['target'] = 1

print("=== Positive Pairs (Real Interactions) ===")
print(f"Unique (customer, vendor) pairs: {len(cv_positives):,}")
print(f"Unique customers:                {cv_positives['customer_id'].nunique():,}")
print(f"Unique vendors:                  {cv_positives['vendor_id'].nunique():,}")
print(f"\nOrder count per pair:")
print(cv_positives['order_count'].describe())

## 4. Generate Negative Samples

For each customer, randomly sample vendors they **did NOT** order from.
Using a **1:1 ratio** (same negatives as positives per customer) to keep classes balanced.

In [ ]:
# --- Generate negatives: (customer, vendor) pairs with no interaction ---
customer_bought = orders.groupby('customer_id')['vendor_id'].apply(set).to_dict()
all_vendor_ids = set(vendors['id'].values)

neg_rows = []
np.random.seed(42)

for cid, bought_set in customer_bought.items():
    not_bought = list(all_vendor_ids - bought_set)
    # 1:1 ratio: match the number of vendors this customer DID order from
    n_pos = len(bought_set)
    n_neg = min(n_pos, len(not_bought))
    if n_neg <= 0:
        continue
    sampled = np.random.choice(not_bought, size=n_neg, replace=False)
    for vid in sampled:
        neg_rows.append({
            'customer_id': cid,
            'vendor_id': vid,
            'order_count': 0,
            'target': 0
        })

neg_df = pd.DataFrame(neg_rows)

# --- Combine positives + negatives ---
full_df = pd.concat([cv_positives, neg_df], ignore_index=True)
full_df = full_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print("=== Combined Dataset ===")
print(f"Positives (target=1): {(full_df['target']==1).sum():,}")
print(f"Negatives (target=0): {(full_df['target']==0).sum():,}")
print(f"Total:                {len(full_df):,}")
print(f"Balance ratio:        {(full_df['target']==0).sum() / (full_df['target']==1).sum():.2f}:1")

print(f"\n=== Target Distribution ===")
print(full_df['target'].value_counts())

## 5. Merge Features

We only use features available at **prediction time** -- no order-specific columns that leak the target.

| Source | Features Used |
|--------|--------------|
| Customer table | gender, status, verified, language |
| Location table | latitude, longitude, location_type |
| Vendor table | category, delivery_charge, rating, prep time, etc. |
| Computed | distance (customer <-> vendor), vendor popularity |

In [ ]:
# --- 5a. Customer demographics ---
full_df = full_df.merge(
    train_customers[['customer_id', 'gender', 'status', 'verified', 'language']],
    on='customer_id', how='left'
)

# --- 5b. Customer primary location (location_number = 0, or first available) ---
primary_loc = (
    train_locations
    .sort_values('location_number')
    .groupby('customer_id')
    .first()
    .reset_index()
    .rename(columns={
        'latitude': 'cust_latitude',
        'longitude': 'cust_longitude'
    })
)
full_df = full_df.merge(
    primary_loc[['customer_id', 'cust_latitude', 'cust_longitude', 'location_type']],
    on='customer_id', how='left'
)

# --- 5c. Vendor attributes ---
vendor_feature_cols = [
    'id', 'vendor_category_en', 'vendor_category_id', 'delivery_charge',
    'serving_distance', 'is_open', 'prepration_time', 'vendor_rating',
    'commission', 'discount_percentage', 'rank', 'open_close_flags',
    'country_id', 'city_id', 'display_orders'
]
vendor_features = vendors[vendor_feature_cols].copy()
vendor_features = vendor_features.rename(columns={
    'id': 'vendor_id',
    'vendor_rating': 'vendor_avg_rating'
})
full_df = full_df.merge(vendor_features, on='vendor_id', how='left')

# --- 5d. Vendor location (for distance calculation) ---
vendor_loc = vendors[['id', 'latitude', 'longitude']].rename(columns={
    'id': 'vendor_id',
    'latitude': 'vendor_latitude',
    'longitude': 'vendor_longitude'
})
full_df = full_df.merge(vendor_loc, on='vendor_id', how='left')

# --- 5e. Computed: Euclidean distance ---
full_df['distance'] = np.sqrt(
    (full_df['cust_latitude'] - full_df['vendor_latitude'])**2 +
    (full_df['cust_longitude'] - full_df['vendor_longitude'])**2
)

# --- 5f. Computed: Vendor popularity (total orders received in training) ---
vendor_pop = orders.groupby('vendor_id')['order_id'].count().reset_index()
vendor_pop.columns = ['vendor_id', 'vendor_popularity']
full_df = full_df.merge(vendor_pop, on='vendor_id', how='left')
full_df['vendor_popularity'] = full_df['vendor_popularity'].fillna(0)

print("=== Dataset After Feature Merge ===")
print(f"Shape: {full_df.shape}")
print(f"Columns: {full_df.columns.tolist()}")

## 6. Feature Selection & Cleaning

In [ ]:
# --- Define feature columns (ONLY prediction-time available features) ---
feature_cols = [
    # Customer demographics
    'gender', 'status', 'verified', 'language', 'location_type',
    # Customer location
    'cust_latitude', 'cust_longitude',
    # Vendor attributes
    'vendor_category_en', 'vendor_category_id', 'delivery_charge',
    'serving_distance', 'is_open', 'prepration_time', 'vendor_avg_rating',
    'commission', 'discount_percentage', 'rank', 'open_close_flags',
    'country_id', 'city_id', 'display_orders',
    # Vendor location
    'vendor_latitude', 'vendor_longitude',
    # Computed features
    'distance', 'vendor_popularity',
]

X = full_df[feature_cols].copy()
y = full_df['target'].copy()

# --- Fill missing values ---
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].fillna('Unknown')
    else:
        X[col] = X[col].fillna(X[col].median())

print("=== Feature Matrix ===")
print(f"Shape: {X.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nFeature dtypes:")
print(X.dtypes.value_counts())
print(f"\nNull counts: {X.isnull().sum().sum()}")

## 7. Encode Categorical Features

In [ ]:
from sklearn.preprocessing import LabelEncoder

trained_encoders = {}
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print("=== Encoding Categorical Features ===")
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    trained_encoders[col] = le
    print(f"  '{col}' -> {len(le.classes_)} unique values")

print(f"\nAll features numeric: {(X.dtypes != 'object').all()}")
print(f"Final X shape: {X.shape}")

## 8. Train XGBClassifier with Cross-Validation

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    n_jobs=-1
)

# --- Cross-validation ---
print("=== 3-Fold Cross-Validation ===")
cv_scores = cross_val_score(model, X, y, cv=3, scoring='roc_auc')
print(f"AUC per fold: {cv_scores}")
print(f"Mean AUC:     {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

# --- Train on full data ---
print("\nTraining on full dataset...")
model.fit(X, y)
print("Model trained successfully!")

## 9. Feature Importance

In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(importance)))
plt.barh(importance['feature'], importance['importance'], color=colors)
plt.xlabel('Importance Score')
plt.title('Feature Importance (XGBoost)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== Feature Importance (Ranked) ===")
print(importance.sort_values('importance', ascending=False).to_string(index=False))

## 10. Prepare Test Data

Build all (customer, location, vendor) combinations for the test set, then merge the **exact same features** used during training.

In [ ]:
# --- Build all test pairs: customer x location x vendor ---
print("Building test pairs...")
test_pairs = []
for _, row in test_locations.iterrows():
    cid = row['customer_id']
    loc_num = row['location_number']
    for vid in vendors['id']:
        test_pairs.append({
            'customer_id': cid,
            'location_number': loc_num,
            'vendor_id': vid
        })

test_df = pd.DataFrame(test_pairs)
print(f"Test pairs: {test_df.shape}")

# --- Merge customer demographics (already deduplicated at load time) ---
test_df = test_df.merge(
    test_customers[['customer_id', 'gender', 'status', 'verified', 'language']],
    on='customer_id', how='left'
)

# --- Merge customer location ---
test_df = test_df.merge(
    test_locations[['customer_id', 'location_number', 'location_type', 'latitude', 'longitude']],
    on=['customer_id', 'location_number'], how='left'
)
test_df = test_df.rename(columns={
    'latitude': 'cust_latitude',
    'longitude': 'cust_longitude'
})

# --- Merge vendor features (reusing same dataframes from training) ---
test_df = test_df.merge(vendor_features, on='vendor_id', how='left')
test_df = test_df.merge(vendor_loc, on='vendor_id', how='left')

# --- Compute distance ---
test_df['distance'] = np.sqrt(
    (test_df['cust_latitude'] - test_df['vendor_latitude'])**2 +
    (test_df['cust_longitude'] - test_df['vendor_longitude'])**2
)

# --- Merge vendor popularity ---
test_df = test_df.merge(vendor_pop, on='vendor_id', how='left')
test_df['vendor_popularity'] = test_df['vendor_popularity'].fillna(0)

print(f"Test dataset shape: {test_df.shape}")
print(f"Expected shape: ({16720 * 100}, ...)")

## 11. Encode Test Features & Predict

In [ ]:
# --- Select the same feature columns as training ---
test_X = test_df[feature_cols].copy()

# --- Fill missing values (same strategy as training) ---
for col in test_X.columns:
    if test_X[col].dtype == 'object':
        test_X[col] = test_X[col].fillna('Unknown')
    else:
        test_X[col] = test_X[col].fillna(test_X[col].median())

# --- Encode categoricals using SAVED encoders from training ---
for col, le in trained_encoders.items():
    if col in test_X.columns:
        known_classes = set(le.classes_)
        fallback = 'Unknown' if 'Unknown' in known_classes else le.classes_[0]
        series_str = test_X[col].astype(str)
        # Replace unseen labels with fallback
        test_X[col] = series_str.where(series_str.isin(known_classes), fallback)
        test_X[col] = le.transform(test_X[col])

# --- Ensure all numeric ---
test_X = test_X.fillna(0)

# --- Predict probabilities ---
print("Predicting on test set...")
test_df['target'] = model.predict_proba(test_X)[:, 1]

print(f"\n=== Prediction Distribution ===")
print(test_df['target'].describe())

## 12. Create Submission File

In [ ]:
# --- Build submission in required format ---
submission = test_df[['customer_id', 'location_number', 'vendor_id', 'target']].copy()

submission['CID X LOC_NUM X VENDOR'] = (
    submission['customer_id'].astype(str) + ' X ' +
    submission['location_number'].astype(str) + ' X ' +
    submission['vendor_id'].astype(str)
)

final_sub = submission[['CID X LOC_NUM X VENDOR', 'target']]

# --- Sanity checks ---
expected_rows = 16720 * 100
print(f"=== Submission Sanity Checks ===")
print(f"Rows: {len(final_sub):,} (expected {expected_rows:,})")
print(f"Duplicate IDs: {final_sub['CID X LOC_NUM X VENDOR'].duplicated().sum()}")
print(f"Null values: {final_sub.isnull().sum().sum()}")
print(f"Columns: {final_sub.columns.tolist()}")

# --- Save ---
final_sub.to_csv('submission.csv', index=False)
print(f"\nSubmission saved to submission.csv!")

print(f"\nSample rows:")
print(final_sub.head(10))

# --- Visualize prediction distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(final_sub['target'], bins=50, edgecolor='black', alpha=0.7, color='#2196F3')
axes[0].set_title('Distribution of Recommendation Scores', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Count')

# Top predictions per customer sample
sample_cid = test_df['customer_id'].iloc[0]
sample_scores = test_df[test_df['customer_id'] == sample_cid].nlargest(10, 'target')
axes[1].barh(
    sample_scores['vendor_id'].astype(str),
    sample_scores['target'],
    color='#4CAF50', edgecolor='black', alpha=0.8
)
axes[1].set_title(f'Top 10 Vendors for Customer {sample_cid}', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recommendation Score')
axes[1].set_ylabel('Vendor ID')

plt.tight_layout()
plt.show()

print("\nDone! Submission file 'submission.csv' is ready.")